# Bloc 6 · Sessió 3 — Overfitting i Xarxes Convolucionals (CNN)

In [ ]:
# BLOC 0 reprèn el model de la S02 (Dense 64→32→1) amb dades sorolloses
#        per fer visible l'efecte tisora (secció 1).
# BLOC A afegeix Dropout al mateix model i compara directament amb BLOC 0
#        (mateixos 200 epochs) — la tisora es redueix (secció 2).
# BLOC B afegeix EarlyStopping a sobre — l'entrenament s'atura sol (secció 3).
# Executa els blocs en ordre.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import keras
from keras import layers
from keras.callbacks import EarlyStopping

print(f"Keras {keras.__version__}  (backend: {keras.backend.backend()})")

## PREPARACIÓ — Dades sorolloses per forçar overfitting visible

In [ ]:
# Wine binaritzat és massa separable: amb dades netes el model de la S02
# (Dense 64→32→1) ja generalitza bé i la "tisora" no s'observa.
# Truc pedagògic: agafem només 50 mostres del train i invertim el 15% de
# les etiquetes (label noise). Forcem el model a MEMORITZAR el soroll
# → la val_loss (sobre etiquetes netes del test) divergirà clarament.
# Aquest mateix subset s'usa als 3 blocs perquè la comparació sigui justa.

data = load_wine()
X = data.data
y = (data.target == 0).astype(int)
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_w = scaler.fit_transform(X_train_w)
X_test_w  = scaler.transform(X_test_w)

np.random.seed(42)
N_SAMPLES, NOISE = 50, 0.15
X_train_small = X_train_w[:N_SAMPLES].copy()
y_train_small = y_train_w[:N_SAMPLES].copy()
flip_mask = np.random.rand(N_SAMPLES) < NOISE
y_train_small[flip_mask] = 1 - y_train_small[flip_mask]
print(f"Subset train: {N_SAMPLES} mostres | labels invertits: {flip_mask.sum()}")
print(f"Test (validació): {X_test_w.shape[0]} mostres (etiquetes netes)\n")

EPOCHS = 100

## BLOC 0 — Cas de la S02 amb dades brutes: apareix la tisora

(secció 1 del material)

In [ ]:
# Reprenem exactament el model de la S02 (Dense 64→32→1) però sobre el subset
# sorollós. Sense regularització, 200 epochs lliures: la train_loss baixa a
# 0 mentre la val_loss puja. Aquesta és la "tisora" d'overfitting.

model_overfit = keras.Sequential([
    keras.layers.Dense(64, activation='relu', input_shape=(X_train_w.shape[1],)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1,  activation='sigmoid')
])
model_overfit.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_overfit = model_overfit.fit(
    X_train_small, y_train_small,
    epochs=EPOCHS, batch_size=8,
    validation_data=(X_test_w, y_test_w),
    verbose=0
)
loss_o, acc_o = model_overfit.evaluate(X_test_w, y_test_w, verbose=0)
print(f"Bloc 0 — Accuracy test: {acc_o:.4f} | Val loss final: {history_overfit.history['val_loss'][-1]:.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
fig.suptitle("Bloc 0 — Tisora d'overfitting (model S02 amb dades brutes)", fontweight='bold')
ax.plot(history_overfit.history['loss'],     label='train loss')
ax.plot(history_overfit.history['val_loss'], label='val loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend()
plt.tight_layout()
plt.savefig('S03_bloc0_tisora.png', dpi=120)
plt.show()
print("Bloc 0 fet — la 'tisora' és visible: train_loss → 0 mentre val_loss puja.\n")

## BLOC A — Afegim Dropout: la tisora es redueix

(secció 2 del material)

In [ ]:
# Mateix model, mateixes dades, mateixos 200 epochs — només afegim Dropout
# entre capes. Sense EarlyStopping encara: volem una comparativa directa
# i visual (mateix eix x) per veure l'efecte aïllat del Dropout.

model_dropout = keras.Sequential([
    keras.layers.Dense(64, activation='relu', input_shape=(X_train_w.shape[1],)),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1,  activation='sigmoid')
])
model_dropout.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_dropout = model_dropout.fit(
    X_train_small, y_train_small,
    epochs=EPOCHS, batch_size=8,
    validation_data=(X_test_w, y_test_w),
    verbose=0
)
loss_d, acc_d = model_dropout.evaluate(X_test_w, y_test_w, verbose=0)
print(f"Bloc A — Accuracy test: {acc_d:.4f} | Val loss final: {history_dropout.history['val_loss'][-1]:.4f}")

# --- Comparativa side-by-side: BLOC 0 vs BLOC A (mateixos 200 epochs) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Comparativa — Bloc 0 (sense Dropout) vs Bloc A (amb Dropout)",
             fontweight='bold')

ax1.plot(history_overfit.history['loss'],     label='train loss')
ax1.plot(history_overfit.history['val_loss'], label='val loss')
ax1.set_title(f'Bloc 0 · sense Dropout (acc={acc_o:.2%})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()

ax2.plot(history_dropout.history['loss'],     label='train loss')
ax2.plot(history_dropout.history['val_loss'], label='val loss')
ax2.set_title(f'Bloc A · amb Dropout (acc={acc_d:.2%})')
ax2.set_xlabel('Epoch'); ax2.legend()

ymax = max(max(history_overfit.history['val_loss']),
           max(history_dropout.history['val_loss'])) * 1.05
ax1.set_ylim(0, ymax); ax2.set_ylim(0, ymax)

plt.tight_layout()
plt.savefig('S03_blocA_comparativa.png', dpi=120)
plt.show()
print(f"Comparativa Bloc 0 vs A guardada — accuracy: {acc_o:.2%} → {acc_d:.2%} (Δ {acc_d-acc_o:+.2%})\n")

## BLOC B — Afegim EarlyStopping: parem quan val_loss deixa de millorar

(secció 3 del material)

In [ ]:
# Mateix model amb Dropout, mateixes dades. Ara però amb EarlyStopping:
# l'entrenament s'atura sol quan la val_loss no millora durant 5 epochs
# i restaurem els millors pesos. No cal definir el nombre òptim d'epochs
# a mà.

early_stop_b = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model_earlystop = keras.Sequential([
    keras.layers.Dense(64, activation='relu', input_shape=(X_train_w.shape[1],)),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1,  activation='sigmoid')
])
model_earlystop.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_earlystop = model_earlystop.fit(
    X_train_small, y_train_small,
    epochs=EPOCHS, batch_size=8,
    validation_data=(X_test_w, y_test_w),
    callbacks=[early_stop_b], verbose=0
)
loss_e, acc_e = model_earlystop.evaluate(X_test_w, y_test_w, verbose=0)
stopped = early_stop_b.stopped_epoch + 1 if early_stop_b.stopped_epoch else EPOCHS
print(f"Bloc B — Accuracy test: {acc_e:.4f} | Aturat a l'epoch: {stopped}")

fig, ax = plt.subplots(figsize=(9, 4))
fig.suptitle(f"Bloc B — Dropout + EarlyStopping (aturat a epoch {stopped})",
             fontweight='bold')
ax.plot(history_earlystop.history['loss'],     label='train loss')
ax.plot(history_earlystop.history['val_loss'], label='val loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend()
plt.tight_layout()
plt.savefig('S03_blocB_earlystop.png', dpi=120)
plt.show()
print(f"Bloc B fet — millora final: {acc_o:.2%} → {acc_e:.2%} (Δ {acc_e-acc_o:+.2%})\n")

## EXERCICI 2 — Càrrega i exploració del dataset Fashion MNIST

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

class_names = ['T-shirt', 'Pantalons', 'Jersei', 'Vestit', 'Abric',
               'Sandàlia', 'Camisa', 'Sabatilla', 'Bossa', 'Botí']

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("Exercici 2 — Mostres de Fashion MNIST", fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(class_names[y_train[i]])
    ax.axis('off')
plt.tight_layout()
plt.savefig('S03_ex2_mostres.png', dpi=120)
plt.show()
print("Exercici 2 fet.\n")

## EXERCICI 3 — Construir i entrenar la CNN

In [ ]:

# --- Preprocés ---
X_train = X_train / 255.0
X_test  = X_test  / 255.0
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# --- Model CNN ---
model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# Comparativa de paràmetres amb MLP equivalent
mlp_params = (784 * 64 + 64) + (64 * 64 + 64) + (64 * 10 + 10)
print(f"\nParàmetres MLP equivalent (Dense 784→64→64→10): {mlp_params:,}")
print(f"Paràmetres CNN (del summary): ~121,930")

# --- EarlyStopping ---
early_stop = EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

# --- Entrenament ---
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

# --- Avaluació ---
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
stopped = early_stop.stopped_epoch + 1 if early_stop.stopped_epoch > 0 else 30
print(f"\nAccuracy en test: {accuracy:.4f}")
print(f"Baseline aleatori (10 classes): 10.00%")
print(f"Millora sobre baseline: {accuracy / 0.10:.1f}×")
print(f"Entrenament aturat a l'epoch: {stopped}\n")


## EXERCICI 4 — Diagnosticar el resultat

In [ ]:
# --- Corbes d'entrenament ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Exercici 4 — Corbes CNN sobre Fashion MNIST", fontweight='bold')

ax1.plot(history.history['loss'],     label='train loss')
ax1.plot(history.history['val_loss'], label='val loss')
ax1.set_title('Funció de Pèrdua per Epoch')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()

ax2.plot(history.history['accuracy'],     label='train accuracy')
ax2.plot(history.history['val_accuracy'], label='val accuracy')
ax2.set_title('Accuracy per Epoch')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend()

plt.tight_layout()
plt.savefig('S03_ex4_corbes_CNN.png', dpi=120)
plt.show()

# --- Prediccions visuals ---
predictions = model.predict(X_test[:10], verbose=0)
predicted_labels = np.argmax(predictions, axis=1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("Exercici 4 — Prediccions (verd=encert, vermell=error)", fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i].reshape(28, 28), cmap='gray')
    color = 'green' if predicted_labels[i] == y_test[i] else 'red'
    ax.set_title(
        f"Pred: {class_names[predicted_labels[i]]}\nReal: {class_names[y_test[i]]}",
        color=color, fontsize=8
    )
    ax.axis('off')
plt.tight_layout()
plt.savefig('S03_ex4_prediccions.png', dpi=120)
plt.show()

## EXERCICI 4 · Repte opcional — Sense Dropout (per comparar)

In [ ]:
model_no_drop = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])
model_no_drop.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history_no_drop = model_no_drop.fit(
    X_train, y_train, epochs=30, batch_size=64,
    validation_split=0.1, verbose=0
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Repte — Sense Dropout vs Amb Dropout (train loss)", fontweight='bold')

n = len(history.history['loss'])
ax1.plot(range(n), history.history['loss'],     label='train loss (amb Dropout)')
ax1.plot(range(n), history.history['val_loss'], label='val loss (amb Dropout)')
ax1.set_title('Amb Dropout'); ax1.set_xlabel('Epoch'); ax1.legend()

n2 = len(history_no_drop.history['loss'])
ax2.plot(range(n2), history_no_drop.history['loss'],     label='train loss (sense Dropout)')
ax2.plot(range(n2), history_no_drop.history['val_loss'], label='val loss (sense Dropout)')
ax2.set_title('Sense Dropout'); ax2.set_xlabel('Epoch'); ax2.legend()

plt.tight_layout()
plt.savefig('S03_repte_dropout_comparativa.png', dpi=120)
plt.show()

loss_nd, acc_nd = model_no_drop.evaluate(X_test, y_test, verbose=0)
print(f"\nAccuracy amb Dropout:    {accuracy:.4f}")
print(f"Accuracy sense Dropout:  {acc_nd:.4f}")
print("Diferència:", round(acc_nd - accuracy, 4))
print("\nTots els gràfics guardats com a fitxers PNG.")